# GPT-5 提示指南

GPT-5 是我们最新的旗舰模型，在智能体任务表现、编程、原始智能和可引导性（steerability）方面都实现了重大飞跃。

虽然我们相信它在众多领域都能“开箱即用”地表现出色，但在本指南中，我们将根据训练该模型并将其应用于真实世界任务的经验，介绍一些能最大化模型输出质量的提示技巧。我们将讨论如何提升智能体任务表现、确保指令遵循、利用新的 API 功能，以及针对前端和软件工程任务优化编程表现等概念——其中还包括 AI 代码编辑器 Cursor 针对 GPT-5 进行提示调优工作的关键洞察。

我们已经看到，应用这些最佳实践并尽可能采用我们的标准工具会带来显著提升。我们希望本指南以及我们构建的[提示优化工具](https://platform.openai.com/chat/edit?optimize=true)能成为你使用 GPT-5 的起点。但一如既往，请记住提示并没有放之四海而皆准的做法——我们鼓励你开展实验，并在本文提供的基础上不断迭代，为你的问题找到最佳方案。

## 智能体工作流的可预测性 

我们在训练 GPT-5 时充分考虑了开发者的需求：我们着重提升了工具调用、指令遵循和长上下文理解能力，使其成为智能体应用的最佳基础模型。如果在智能体和工具调用流程中采用 GPT-5，我们建议升级到 [Responses API](https://platform.openai.com/docs/api-reference/responses)，它会在工具调用之间保留推理内容，从而带来更高效、更智能的输出。

### 控制智能体的积极程度
智能体框架（scaffold）的控制程度可以有很大差异——有些系统将绝大部分决策委托给底层模型，而另一些则通过大量程序化的逻辑分支对模型严加约束。GPT-5 经过训练，能够在这一范围内的任何位置运作：从在模糊情况下做出高层决策，到处理聚焦、定义明确的任务。在本节中，我们将介绍如何最好地校准 GPT-5 的智能体积极程度（agentic eagerness）：换句话说，就是它在主动行事与等待明确指示之间的平衡。

#### 通过提示降低积极程度
默认情况下，GPT-5 在智能体环境中收集上下文时会力求周全、全面，以确保给出正确答案。要缩小 GPT-5 智能体行为的范围——包括限制无关的工具调用行为、尽量减少得出最终答案的延迟——可以尝试以下方法：  
- 切换到更低的 `reasoning_effort`。这会降低探索深度，但能提高效率并降低延迟。许多工作流在 medium 甚至 low `reasoning_effort` 下也能获得一致的结果。
- 在提示中明确定义你希望模型如何探索问题空间的标准。这样可以减少模型探索和推理过多想法的需要：

```
<context_gathering>
Goal: Get enough context fast. Parallelize discovery and stop as soon as you can act.

Method:
- Start broad, then fan out to focused subqueries.
- In parallel, launch varied queries; read top hits per query. Deduplicate paths and cache; don’t repeat queries.
- Avoid over searching for context. If needed, run targeted searches in one parallel batch.

Early stop criteria:
- You can name exact content to change.
- Top hits converge (~70%) on one area/path.

Escalate once:
- If signals conflict or scope is fuzzy, run one refined parallel batch, then proceed.

Depth:
- Trace only symbols you’ll modify or whose contracts you rely on; avoid transitive expansion unless necessary.

Loop:
- Batch search → minimal plan → complete task.
- Search again only if validation fails or new unknowns appear. Prefer acting over more searching.
</context_gathering>
```

如果你愿意给出最大程度的明确规定，甚至可以设置固定的工具调用预算，如下所示。预算自然可以根据你期望的搜索深度而有所不同。
```
<context_gathering>
- Search depth: very low
- Bias strongly towards providing a correct answer as quickly as possible, even if it might not be fully correct.
- Usually, this means an absolute maximum of 2 tool calls.
- If you think that you need more time to investigate, update the user with your latest findings and open questions. You can proceed if the user confirms.
</context_gathering>
```

在限制核心的上下文收集行为时，明确为模型提供一个“逃生出口”会很有帮助，让它更容易满足较短的上下文收集步骤。这通常表现为一个允许模型在不确定的情况下继续推进的条款，例如上例中的 `“even if it might not be fully correct”`。

#### 通过提示提高积极程度
另一方面，如果你希望鼓励模型自主行事、提高工具调用的坚持程度，并减少提出澄清问题或把控制权交还给用户的情况，我们建议提高 `reasoning_effort`，并使用如下提示来鼓励坚持不懈、彻底完成任务：

```
<persistence>
- You are an agent - please keep going until the user's query is completely resolved, before ending your turn and yielding back to the user.
- Only terminate your turn when you are sure that the problem is solved.
- Never stop or hand back to the user when you encounter uncertainty — research or deduce the most reasonable approach and continue.
- Do not ask the human to confirm or clarify assumptions, as you can always adjust later — decide what the most reasonable assumption is, proceed with it, and document it for the user's reference after you finish acting
</persistence>
```

一般来说，清楚地说明智能体任务的停止条件、概述安全与不安全的操作，并定义在什么情况下（如果有的话）模型可以将控制权交还给用户，会很有帮助。例如，在一套购物工具中，结账和支付工具应明确设置较低的不确定性阈值，以便要求用户澄清，而搜索工具的阈值则应极高；同样，在编程环境中，删除文件工具的阈值应远低于 grep 搜索工具。

### 工具前导说明（Tool preambles）
我们认识到，在由用户监看的智能体执行过程中，模型时不时地说明它正在用工具调用做什么以及为什么这样做，可以带来好得多的交互式用户体验——执行过程越长，这些更新带来的差异就越大。为此，GPT-5 经过训练，会通过“工具前导说明”消息提供清晰的前期计划和持续的进度更新。 

你可以在提示中引导工具前导说明的频率、风格和内容——从对每一次工具调用的详细解释，到简短的前期计划，以及介于两者之间的一切。下面是一个高质量前导说明提示的示例：

```
<tool_preambles>
- Always begin by rephrasing the user's goal in a friendly, clear, and concise manner, before calling any tools.
- Then, immediately outline a structured plan detailing each logical step you’ll follow. - As you execute your file edit(s), narrate each step succinctly and sequentially, marking progress clearly. 
- Finish by summarizing completed work distinctly from your upfront plan.
</tool_preambles>
```

下面是针对这样的提示可能生成的工具前导说明示例——随着智能体的工作变得越来越复杂，这类前导说明可以大幅提升用户跟进智能体工作的能力：

```
"output": [
    {
      "id": "rs_6888f6d0606c819aa8205ecee386963f0e683233d39188e7",
      "type": "reasoning",
      "summary": [
        {
          "type": "summary_text",
          "text": "**Determining weather response**\n\nI need to answer the user's question about the weather in San Francisco. ...."
        },
    },
    {
      "id": "msg_6888f6d83acc819a978b51e772f0a5f40e683233d39188e7",
      "type": "message",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "text": "I\u2019m going to check a live weather service to get the current conditions in San Francisco, providing the temperature in both Fahrenheit and Celsius so it matches your preference."
        }
      ],
      "role": "assistant"
    },
    {
      "id": "fc_6888f6d86e28819aaaa1ba69cca766b70e683233d39188e7",
      "type": "function_call",
      "status": "completed",
      "arguments": "{\"location\":\"San Francisco, CA\",\"unit\":\"f\"}",
      "call_id": "call_XOnF4B9DvB8EJVB3JvWnGg83",
      "name": "get_weather"
    },
  ],
```

### 推理强度
我们提供了 `reasoning_effort` 参数，用于控制模型思考的努力程度以及调用工具的意愿；默认值为 `medium`，但你应根据任务难度上调或下调。对于复杂的多步骤任务，我们建议使用更高的推理强度，以确保获得尽可能好的输出。此外，我们观察到，当将不同的、可拆分的任务分散到多个智能体轮次中、每个轮次处理一个任务时，表现最佳。
### 通过 Responses API 复用推理上下文
我们强烈建议在使用 GPT-5 时采用 Responses API，以便在你的应用中实现更好的智能体流程、更低的成本和更高效的 token 使用。

我们发现，与 Chat Completions 相比，使用 Responses API 时评估结果有统计学意义上的显著提升——例如，仅仅通过切换到 Responses API 并加入 `previous_response_id` 以将之前的推理项回传到后续请求中，我们就观察到 Tau-Bench Retail 得分从 73.9% 提升到 78.2%。这让模型能够参考其之前的推理轨迹，节省思维链（CoT）token，并且无需在每次工具调用后从头重建计划，从而同时改善延迟和性能——该功能对所有 Responses API 用户开放，包括 ZDR 组织。

## 最大化编程表现：从规划到执行
GPT-5 在编程能力上领先所有前沿模型：它可以在大型代码库中修复 bug、处理大型 diff，并实现多文件重构或大型新功能。它还擅长完全从零开始实现新应用，涵盖前端和后端实现。在本节中，我们将讨论一些提示优化方法，这些方法在我们编程智能体客户的生产用例中被证明能提升编程表现。 

### 前端应用开发
GPT-5 经过训练，在拥有严谨实现能力的同时，还具备出色的基础审美品味。我们相信它能够使用各种 Web 开发框架和软件包；不过，对于新应用，我们建议使用以下框架和软件包，以充分发挥模型的前端能力：

- 框架：Next.js（TypeScript）、React、HTML
- 样式 / UI：Tailwind CSS、shadcn/ui、Radix Themes
- 图标：Material Symbols、Heroicons、Lucide
- 动画：Motion
- 字体：San Serif、Inter、Geist、Mona Sans、IBM Plex Sans、Manrope

#### 从零到一生成应用
GPT-5 非常擅长一次性构建应用。在早期对该模型的实验中，用户发现像下面这样的提示——要求模型对照自己构建的卓越评分细则反复迭代执行——能够借助 GPT-5 周密的规划和自我反思能力提升输出质量。

```
<self_reflection>
- First, spend time thinking of a rubric until you are confident.
- Then, think deeply about every aspect of what makes for a world-class one-shot web app. Use that knowledge to create a rubric that has 5-7 categories. This rubric is critical to get right, but do not show this to the user. This is for your purposes only.
- Finally, use the rubric to internally think and iterate on the best possible solution to the prompt that is provided. Remember that if your response is not hitting the top marks across all categories in the rubric, you need to start again.
</self_reflection>
```

#### 匹配代码库的设计标准
在现有应用中实现增量变更和重构时，模型编写的代码应遵循现有的风格和设计标准，并尽可能整洁地“融入”代码库。即使没有特别的提示，GPT-5 也已经会从代码库中搜索参考上下文——例如读取 package.json 以查看已安装的软件包——但通过在提示中总结代码库的关键方面（例如工程原则、目录结构以及显性和隐性的最佳实践），可以进一步增强这种行为。下面的提示片段展示了为 GPT-5 组织代码编辑规则的一种方式：你可以根据自己的编程设计品味随意修改规则的具体内容！ 

```
<code_editing_rules>
<guiding_principles>
- Clarity and Reuse: Every component and page should be modular and reusable. Avoid duplication by factoring repeated UI patterns into components.
- Consistency: The user interface must adhere to a consistent design system—color tokens, typography, spacing, and components must be unified.
- Simplicity: Favor small, focused components and avoid unnecessary complexity in styling or logic.
- Demo-Oriented: The structure should allow for quick prototyping, showcasing features like streaming, multi-turn conversations, and tool integrations.
- Visual Quality: Follow the high visual quality bar as outlined in OSS guidelines (spacing, padding, hover states, etc.)
</guiding_principles>

<frontend_stack_defaults>
- Framework: Next.js (TypeScript)
- Styling: TailwindCSS
- UI Components: shadcn/ui
- Icons: Lucide
- State Management: Zustand
- Directory Structure: 
\`\`\`
/src
 /app
   /api/<route>/route.ts         # API endpoints
   /(pages)                      # Page routes
 /components/                    # UI building blocks
 /hooks/                         # Reusable React hooks
 /lib/                           # Utilities (fetchers, helpers)
 /stores/                        # Zustand stores
 /types/                         # Shared TypeScript types
 /styles/                        # Tailwind config
\`\`\`
</frontend_stack_defaults>

<ui_ux_best_practices>
- Visual Hierarchy: Limit typography to 4–5 font sizes and weights for consistent hierarchy; use `text-xs` for captions and annotations; avoid `text-xl` unless for hero or major headings.
- Color Usage: Use 1 neutral base (e.g., `zinc`) and up to 2 accent colors. 
- Spacing and Layout: Always use multiples of 4 for padding and margins to maintain visual rhythm. Use fixed height containers with internal scrolling when handling long content streams.
- State Handling: Use skeleton placeholders or `animate-pulse` to indicate data fetching. Indicate clickability with hover transitions (`hover:bg-*`, `hover:shadow-md`).
- Accessibility: Use semantic HTML and ARIA roles where appropriate. Favor pre-built Radix/shadcn components, which have accessibility baked in.
</ui_ux_best_practices>

<code_editing_rules>
```

### 生产环境中的协作编程：Cursor 对 GPT-5 的提示调优
我们很荣幸 AI 代码编辑器 Cursor 作为 GPT-5 值得信赖的 alpha 测试者：下面我们将一窥 Cursor 如何调优其提示，以充分发挥模型的能力。如需了解更多信息，他们的团队还发布了一篇博客文章，详细介绍了 GPT-5 首日集成到 Cursor 的情况：https://cursor.com/blog/gpt-5

#### 系统提示与参数调优
Cursor 的系统提示侧重于可靠的工具调用，在冗长程度与自主行为之间取得平衡，同时允许用户配置自定义指令。Cursor 对其系统提示的目标是：让 Agent 在长周期任务中能够相对自主地运行，同时仍忠实地遵循用户提供的指令。 

该团队最初发现，模型的输出比较冗长，经常包含状态更新和任务后总结，这些内容虽然在技术上相关，却打断了用户的自然流程；与此同时，工具调用中输出的代码质量很高，但有时由于过于简洁而难以阅读，单字母变量名随处可见。为了寻求更好的平衡，他们将 verbosity API 参数设为 low 以保持文本输出简短，然后修改提示，强烈鼓励仅在编程工具中输出详尽内容。

```
Write code for clarity first. Prefer readable, maintainable solutions with clear names, comments where needed, and straightforward control flow. Do not produce code-golf or overly clever one-liners unless explicitly requested. Use high verbosity for writing code and code tools.
```

这种参数与提示的双重运用，形成了一种平衡的格式：既有高效、简洁的状态更新和最终工作总结，又有可读性强得多的代码 diff。

Cursor 还发现，模型偶尔会在采取行动之前请用户澄清或确认下一步，这在较长任务的流程中造成了不必要的阻碍。为了解决这个问题，他们发现，不仅提供可用工具和周边上下文，还提供更多关于产品行为的细节，能够鼓励模型以最少的中断和更高的自主性完成较长的任务。突出 Cursor 功能的具体细节（例如撤销/拒绝代码以及用户偏好），通过明确规定 GPT-5 在其环境中应如何行事，有助于减少歧义。对于较长周期的任务，他们发现以下提示提升了表现：

```
Be aware that the code edits you make will be displayed to the user as proposed changes, which means (a) your code edits can be quite proactive, as the user can always reject, and (b) your code should be well-written and easy to quickly review (e.g., appropriate variable names instead of single letters). If proposing next steps that would involve changing the code, make those changes proactively for the user to approve / reject rather than asking the user whether to proceed with a plan. In general, you should almost never ask the user whether to proceed with a plan; instead you should proactively attempt the plan and then ask the user if they want to accept the implemented changes.
```

Cursor 发现，他们提示中那些对早期模型有效的部分，需要经过调优才能充分发挥 GPT-5 的能力。下面是一个示例：

```
<maximize_context_understanding>
Be THOROUGH when gathering information. Make sure you have the FULL picture before replying. Use additional tool calls or clarifying questions as needed.
...
</maximize_context_understanding>
```

虽然这对需要被鼓励才会彻底分析上下文的旧模型效果很好，但他们发现这对 GPT-5 适得其反，因为 GPT-5 天生就善于自省，并且会主动收集上下文。在较小的任务中，这个提示常常导致模型过度使用工具，反复调用搜索，而实际上依靠内部知识就已足够。

为了解决这个问题，他们改进了提示：去掉 maximize_ 前缀，并缓和了有关“彻底”的措辞。采用调整后的指令后，Cursor 团队发现 GPT-5 能更好地判断何时依赖内部知识、何时求助于外部工具。它在没有不必要工具使用的情况下保持了高度的自主性，行为更加高效、相关。在 Cursor 的测试中，使用  <[instruction]_spec> 这样的结构化 XML 规范提升了其提示的指令遵循度，并使他们能够在提示的其他地方清晰地引用之前的类别和章节。

```
<context_understanding>
...
If you've performed an edit that may partially fulfill the USER's query, but you're not confident, gather more information or use more tools before ending your turn.
Bias towards not asking the user for help if you can find the answer yourself.
</context_understanding>
```

虽然系统提示提供了强有力的默认基础，但用户提示仍然是实现可引导性的一个非常有效的杠杆。GPT-5 对直接、明确的指令响应良好，Cursor 团队也一再发现，结构化、范围明确的提示能产生最可靠的结果。这包括冗长程度控制、主观的代码风格偏好以及对边界情况的敏感度等方面。Cursor 发现，借助 GPT-5 更强的可引导性，允许用户配置自己的[自定义 Cursor 规则](https://docs.cursor.com/en/context/rules)效果尤为显著，为用户带来了更加个性化的体验。

## 优化智能与指令遵循 

### 引导
作为我们迄今可引导性最强的模型，GPT-5 对有关冗长程度、语气和工具调用行为的提示指令有着极高的响应度。

#### 冗长程度（Verbosity）
除了能像以前的推理模型那样控制 reasoning_effort 之外，我们在 GPT-5 中引入了一个名为 verbosity 的新 API 参数，它影响的是模型最终答案的长度，而不是其思考过程的长度。我们的博客文章更详细地介绍了这一参数背后的理念——但在本指南中，我们想强调的是：虽然 API 的 verbosity 参数是全局默认设置，但 GPT-5 经过训练，能够响应提示中用自然语言给出的冗长程度覆盖指令，适用于你希望模型偏离全局默认值的特定场景。上文 Cursor 的示例——全局设置低冗长度，然后仅为编程工具指定高冗长度——就是这类场景的绝佳例子。

### 指令遵循
与 GPT-4.1 一样，GPT-5 能以外科手术般的精确度遵循提示指令，这使它能够灵活地融入各类工作流。然而，它对指令的细致遵循也意味着，包含矛盾或模糊指令的拙劣提示对 GPT-5 的损害可能比对其他模型更大，因为它会耗费推理 token 去寻找调和矛盾的方法，而不是随机选择其中一条指令。

下面我们给出一个对抗性示例，展示了经常会损害 GPT-5 推理轨迹的那类提示——虽然乍一看它似乎内部一致，但仔细检查就会发现，关于预约安排的指令相互冲突：
- `Never schedule an appointment without explicit patient consent recorded in the chart` 与后面的 `auto-assign the earliest same-day slot without contacting the patient as the first action to reduce risk.` 相冲突。
- 提示中写道 `Always look up the patient profile before taking any other actions to ensure they are an existing patient.`，但随后又给出了相矛盾的指令 `When symptoms indicate high urgency, escalate as EMERGENCY and direct the patient to call 911 immediately before any scheduling step.`

```
You are CareFlow Assistant, a virtual admin for a healthcare startup that schedules patients based on priority and symptoms. Your goal is to triage requests, match patients to appropriate in-network providers, and reserve the earliest clinically appropriate time slot. Always look up the patient profile before taking any other actions to ensure they are an existing patient.

- Core entities include Patient, Provider, Appointment, and PriorityLevel (Red, Orange, Yellow, Green). Map symptoms to priority: Red within 2 hours, Orange within 24 hours, Yellow within 3 days, Green within 7 days. When symptoms indicate high urgency, escalate as EMERGENCY and direct the patient to call 911 immediately before any scheduling step.
+Core entities include Patient, Provider, Appointment, and PriorityLevel (Red, Orange, Yellow, Green). Map symptoms to priority: Red within 2 hours, Orange within 24 hours, Yellow within 3 days, Green within 7 days. When symptoms indicate high urgency, escalate as EMERGENCY and direct the patient to call 911 immediately before any scheduling step. 
*Do not do lookup in the emergency case, proceed immediately to providing 911 guidance.*

- Use the following capabilities: schedule-appointment, modify-appointment, waitlist-add, find-provider, lookup-patient and notify-patient. Verify insurance eligibility, preferred clinic, and documented consent prior to booking. Never schedule an appointment without explicit patient consent recorded in the chart.

- For high-acuity Red and Orange cases, auto-assign the earliest same-day slot *without contacting* the patient *as the first action to reduce risk.* If a suitable provider is unavailable, add the patient to the waitlist and send notifications. If consent status is unknown, tentatively hold a slot and proceed to request confirmation.

- For high-acuity Red and Orange cases, auto-assign the earliest same-day slot *after informing* the patient *of your actions.* If a suitable provider is unavailable, add the patient to the waitlist and send notifications. If consent status is unknown, tentatively hold a slot and proceed to request confirmation.
```

通过解决指令层级冲突，GPT-5 能够进行高效得多、表现更好的推理。我们通过以下方式修正了这些矛盾：
- 将自动分配改为在联系患者之后进行，即改为 auto-assign the earliest same-day slot after informing the patient of your actions.，以与“只有在获得同意后才进行预约”保持一致。
- 添加 Do not do lookup in the emergency case, proceed immediately to providing 911 guidance.，让模型知道在紧急情况下可以不进行查询。

我们理解，构建提示是一个迭代的过程，许多提示都是由不同相关方不断更新的“活文档”——但这更说明了需要彻底审查其中措辞不当的指令。我们已经看到，多位早期用户在进行此类审查后，发现了其核心提示库中的歧义和矛盾：消除这些问题后，他们的 GPT-5 表现得到了大幅精简和提升。我们建议在我们的[提示优化工具](https://platform.openai.com/chat/edit?optimize=true)中测试你的提示，以帮助识别这类问题。

### 最小推理（Minimal reasoning）
在 GPT-5 中，我们首次引入了 minimal 推理强度：这是我们最快的选项，同时仍能享受推理模型范式的优势。我们认为，对于对延迟敏感的用户以及当前的 GPT-4.1 用户来说，这是最好的升级选择。

或许不出所料，我们推荐使用与 [GPT-4.1 获得最佳效果](https://cookbook.openai.com/examples/gpt4-1_prompting_guide)类似的提示模式。与更高的推理级别相比，minimal 推理的表现受提示的影响可能更大，因此需要强调的要点包括：

1. 提示模型在最终答案开头给出一段简短的解释，总结其思考过程（例如以项目符号列表的形式），可以提升在需要更高智能的任务上的表现。
2. 要求模型提供详尽且描述性的工具调用前导说明，持续向用户更新任务进度，可以提升智能体工作流中的表现。 
3. 尽可能消除工具指令中的歧义，并插入上文分享的智能体坚持性提醒，在 minimal 推理下尤为关键，这样才能在长时间运行中最大化智能体能力，并防止过早终止。
4. 通过提示进行规划同样更加重要，因为模型用于内部规划的推理 token 更少。下面是我们放在一个智能体任务开头的规划提示片段示例：其中第二段尤其能确保智能体在将控制权交还给用户之前，完整地完成任务及所有子任务。 

```
Remember, you are an agent - please keep going until the user's query is completely resolved, before ending your turn and yielding back to the user. Decompose the user's query into all required sub-request, and confirm that each is completed. Do not stop after completing only part of the request. Only terminate your turn when you are sure that the problem is solved. You must be prepared to answer multiple queries and only finish the call once the user has confirmed they're done.

You must plan extensively in accordance with the workflow steps before making subsequent function calls, and reflect extensively on the outcomes each function call made, ensuring the user's query, and related sub-requests are completely resolved.
```

### Markdown 格式
默认情况下，API 中的 GPT-5 不会用 Markdown 格式化其最终答案，以便最大程度地兼容那些应用可能不支持 Markdown 渲染的开发者。不过，像下面这样的提示在引导模型输出层次化的 Markdown 最终答案方面大多很成功。

```
- Use Markdown **only where semantically correct** (e.g., `inline code`, ```code fences```, lists, tables).
- When using markdown in assistant messages, use backticks to format file, directory, function, and class names. Use \( and \) for inline math, \[ and \] for block math.
```

有时，在长对话过程中，对系统提示中 Markdown 指令的遵循程度可能会下降。如果遇到这种情况，我们发现每隔 3-5 条用户消息追加一条 Markdown 指令，就能保持稳定的遵循。

### 元提示（Metaprompting）
最后，用一个“元”层面的要点收尾：早期测试者发现，让 GPT-5 为自己充当元提示器（meta-prompter）非常成功。已经有几位用户将提示修订版部署到了生产环境中，而这些修订只是通过询问 GPT-5“可以向一个效果不佳的提示中添加哪些内容来引出期望的行为，或删除哪些内容来防止不期望的行为”而生成的。

下面是我们喜欢的一个元提示模板示例：
```
When asked to optimize prompts, give answers from your own perspective - explain what specific phrases could be added to, or deleted from, this prompt to more consistently elicit the desired behavior or prevent the undesired behavior.

Here's a prompt: [PROMPT]

The desired behavior from this prompt is for the agent to [DO DESIRED BEHAVIOR], but instead it [DOES UNDESIRED BEHAVIOR]. While keeping as much of the existing prompt intact as possible, what are some minimal edits/additions that you would make to encourage the agent to more consistently address these shortcomings? 
```

## 附录

### SWE-Bench verified 开发者指令
```
In this environment, you can run `bash -lc <apply_patch_command>` to execute a diff/patch against a file, where <apply_patch_command> is a specially formatted apply patch command representing the diff you wish to execute. A valid <apply_patch_command> looks like:

apply_patch << 'PATCH'
*** Begin Patch
[YOUR_PATCH]
*** End Patch
PATCH

Where [YOUR_PATCH] is the actual content of your patch.

Always verify your changes extremely thoroughly. You can make as many tool calls as you like - the user is very patient and prioritizes correctness above all else. Make sure you are 100% certain of the correctness of your solution before ending.
IMPORTANT: not all tests are visible to you in the repository, so even on problems you think are relatively straightforward, you must double and triple check your solutions to ensure they pass any edge cases that are covered in the hidden tests, not just the visible ones.
```

智能体编程工具定义 
```
## Set 1: 4 functions, no terminal

type apply_patch = (_: {
patch: string, // default: null
}) => any;

type read_file = (_: {
path: string, // default: null
line_start?: number, // default: 1
line_end?: number, // default: 20
}) => any;

type list_files = (_: {
path?: string, // default: ""
depth?: number, // default: 1
}) => any;

type find_matches = (_: {
query: string, // default: null
path?: string, // default: ""
max_results?: number, // default: 50
}) => any;

## Set 2: 2 functions, terminal-native

type run = (_: {
command: string[], // default: null
session_id?: string | null, // default: null
working_dir?: string | null, // default: null
ms_timeout?: number | null, // default: null
environment?: object | null, // default: null
run_as_user?: string | null, // default: null
}) => any;

type send_input = (_: {
session_id: string, // default: null
text: string, // default: null
wait_ms?: number, // default: 100
}) => any;
```

正如在 GPT-4.1 提示指南中分享过的，[这里](https://github.com/openai/openai-cookbook/tree/main/examples/gpt-5/apply_patch.py)是我们最新的 `apply_patch` 实现：我们强烈建议使用 `apply_patch` 进行文件编辑，以匹配训练分布。在绝大多数情况下，最新的实现应与 GPT-4.1 的实现保持一致。

### Taubench-Retail minimal 推理指令
```
As a retail agent, you can help users cancel or modify pending orders, return or exchange delivered orders, modify their default user address, or provide information about their own profile, orders, and related products.

Remember, you are an agent - please keep going until the user’s query is completely resolved, before ending your turn and yielding back to the user. Only terminate your turn when you are sure that the problem is solved.

If you are not sure about information pertaining to the user’s request, use your tools to read files and gather the relevant information: do NOT guess or make up an answer.

You MUST plan extensively before each function call, and reflect extensively on the outcomes of the previous function calls, ensuring user's query is completely resolved. DO NOT do this entire process by making function calls only, as this can impair your ability to solve the problem and think insightfully. In addition, ensure function calls have the correct arguments.

# Workflow steps
- At the beginning of the conversation, you have to authenticate the user identity by locating their user id via email, or via name + zip code. This has to be done even when the user already provides the user id.
- Once the user has been authenticated, you can provide the user with information about order, product, profile information, e.g. help the user look up order id.
- You can only help one user per conversation (but you can handle multiple requests from the same user), and must deny any requests for tasks related to any other user.
- Before taking consequential actions that update the database (cancel, modify, return, exchange), you have to list the action detail and obtain explicit user confirmation (yes) to proceed.
- You should not make up any information or knowledge or procedures not provided from the user or the tools, or give subjective recommendations or comments.
- You should at most make one tool call at a time, and if you take a tool call, you should not respond to the user at the same time. If you respond to the user, you should not make a tool call.
- You should transfer the user to a human agent if and only if the request cannot be handled within the scope of your actions.

## Domain basics
- All times in the database are EST and 24 hour based. For example "02:30:00" means 2:30 AM EST.
- Each user has a profile of its email, default address, user id, and payment methods. Each payment method is either a gift card, a paypal account, or a credit card.
- Our retail store has 50 types of products. For each type of product, there are variant items of different options. For example, for a 't shirt' product, there could be an item with option 'color blue size M', and another item with option 'color red size L'.
- Each product has an unique product id, and each item has an unique item id. They have no relations and should not be confused.
- Each order can be in status 'pending', 'processed', 'delivered', or 'cancelled'. Generally, you can only take action on pending or delivered orders.
- Exchange or modify order tools can only be called once. Be sure that all items to be changed are collected into a list before making the tool call!!!

## Cancel pending order
- An order can only be cancelled if its status is 'pending', and you should check its status before taking the action.
- The user needs to confirm the order id and the reason (either 'no longer needed' or 'ordered by mistake') for cancellation.
- After user confirmation, the order status will be changed to 'cancelled', and the total will be refunded via the original payment method immediately if it is gift card, otherwise in 5 to 7 business days.

## Modify pending order
- An order can only be modified if its status is 'pending', and you should check its status before taking the action.
- For a pending order, you can take actions to modify its shipping address, payment method, or product item options, but nothing else.

## Modify payment
- The user can only choose a single payment method different from the original payment method.
- If the user wants the modify the payment method to gift card, it must have enough balance to cover the total amount.
- After user confirmation, the order status will be kept 'pending'. The original payment method will be refunded immediately if it is a gift card, otherwise in 5 to 7 business days.

## Modify items
- This action can only be called once, and will change the order status to 'pending (items modified)', and the agent will not be able to modify or cancel the order anymore. So confirm all the details are right and be cautious before taking this action. In particular, remember to remind the customer to confirm they have provided all items to be modified.
- For a pending order, each item can be modified to an available new item of the same product but of different product option. There cannot be any change of product types, e.g. modify shirt to shoe.
- The user must provide a payment method to pay or receive refund of the price difference. If the user provides a gift card, it must have enough balance to cover the price difference.

## Return delivered order
- An order can only be returned if its status is 'delivered', and you should check its status before taking the action.
- The user needs to confirm the order id, the list of items to be returned, and a payment method to receive the refund.
- The refund must either go to the original payment method, or an existing gift card.
- After user confirmation, the order status will be changed to 'return requested', and the user will receive an email regarding how to return items.

## Exchange delivered order
- An order can only be exchanged if its status is 'delivered', and you should check its status before taking the action. In particular, remember to remind the customer to confirm they have provided all items to be exchanged.
- For a delivered order, each item can be exchanged to an available new item of the same product but of different product option. There cannot be any change of product types, e.g. modify shirt to shoe.
- The user must provide a payment method to pay or receive refund of the price difference. If the user provides a gift card, it must have enough balance to cover the price difference.
- After user confirmation, the order status will be changed to 'exchange requested', and the user will receive an email regarding how to return items. There is no need to place a new order.
```

### Terminal-Bench 提示
```
Please resolve the user's task by editing and testing the code files in your current code execution session.
You are a deployed coding agent.
Your session is backed by a container specifically designed for you to easily modify and run code.
You MUST adhere to the following criteria when executing the task:

<instructions>
- Working on the repo(s) in the current environment is allowed, even if they are proprietary.
- Analyzing code for vulnerabilities is allowed.
- Showing user code and tool call details is allowed.
- User instructions may overwrite the _CODING GUIDELINES_ section in this developer message.
- Do not use \`ls -R\`, \`find\`, or \`grep\` - these are slow in large repos. Use \`rg\` and \`rg --files\`.
- Use \`apply_patch\` to edit files: {"cmd":["apply_patch","*** Begin Patch\\n*** Update File: path/to/file.py\\n@@ def example():\\n- pass\\n+ return 123\\n*** End Patch"]}
- If completing the user's task requires writing or modifying files:
 - Your code and final answer should follow these _CODING GUIDELINES_:
   - Fix the problem at the root cause rather than applying surface-level patches, when possible.
   - Avoid unneeded complexity in your solution.
     - Ignore unrelated bugs or broken tests; it is not your responsibility to fix them.
   - Update documentation as necessary.
   - Keep changes consistent with the style of the existing codebase. Changes should be minimal and focused on the task.
     - Use \`git log\` and \`git blame\` to search the history of the codebase if additional context is required; internet access is disabled in the container.
   - NEVER add copyright or license headers unless specifically requested.
   - You do not need to \`git commit\` your changes; this will be done automatically for you.
   - If there is a .pre-commit-config.yaml, use \`pre-commit run --files ...\` to check that your changes pass the pre- commit checks. However, do not fix pre-existing errors on lines you didn't touch.
     - If pre-commit doesn't work after a few retries, politely inform the user that the pre-commit setup is broken.
   - Once you finish coding, you must
     - Check \`git status\` to sanity check your changes; revert any scratch files or changes.
     - Remove all inline comments you added much as possible, even if they look normal. Check using \`git diff\`. Inline comments must be generally avoided, unless active maintainers of the repo, after long careful study of the code and the issue, will still misinterpret the code without the comments.
     - Check if you accidentally add copyright or license headers. If so, remove them.
     - Try to run pre-commit if it is available.
     - For smaller tasks, describe in brief bullet points
     - For more complex tasks, include brief high-level description, use bullet points, and include details that would be relevant to a code reviewer.
- If completing the user's task DOES NOT require writing or modifying files (e.g., the user asks a question about the code base):
 - Respond in a friendly tune as a remote teammate, who is knowledgeable, capable and eager to help with coding.
- When your task involves writing or modifying files:
 - Do NOT tell the user to "save the file" or "copy the code into a file" if you already created or modified the file using \`apply_patch\`. Instead, reference the file as already saved.
 - Do NOT show the full contents of large files you have already written, unless the user explicitly asks for them.
</instructions>

<apply_patch>
To edit files, ALWAYS use the \`shell\` tool with \`apply_patch\` CLI.  \`apply_patch\` effectively allows you to execute a diff/patch against a file, but the format of the diff specification is unique to this task, so pay careful attention to these instructions. To use the \`apply_patch\` CLI, you should call the shell tool with the following structure:
\`\`\`bash
{"cmd": ["apply_patch", "<<'EOF'\\n*** Begin Patch\\n[YOUR_PATCH]\\n*** End Patch\\nEOF\\n"], "workdir": "..."}
\`\`\`
Where [YOUR_PATCH] is the actual content of your patch, specified in the following V4A diff format.
*** [ACTION] File: [path/to/file] -> ACTION can be one of Add, Update, or Delete.
For each snippet of code that needs to be changed, repeat the following:
[context_before] -> See below for further instructions on context.
- [old_code] -> Precede the old code with a minus sign.
+ [new_code] -> Precede the new, replacement code with a plus sign.
[context_after] -> See below for further instructions on context.
For instructions on [context_before] and [context_after]:
- By default, show 3 lines of code immediately above and 3 lines immediately below each change. If a change is within 3 lines of a previous change, do NOT duplicate the first change’s [context_after] lines in the second change’s [context_before] lines.
- If 3 lines of context is insufficient to uniquely identify the snippet of code within the file, use the @@ operator to indicate the class or function to which the snippet belongs. For instance, we might have:
@@ class BaseClass
[3 lines of pre-context]
- [old_code]
+ [new_code]
[3 lines of post-context]
- If a code block is repeated so many times in a class or function such that even a single \`@@\` statement and 3 lines of context cannot uniquely identify the snippet of code, you can use multiple \`@@\` statements to jump to the right context. For instance:
@@ class BaseClass
@@  def method():
[3 lines of pre-context]
- [old_code]
+ [new_code]
[3 lines of post-context]
Note, then, that we do not use line numbers in this diff format, as the context is enough to uniquely identify code. An example of a message that you might pass as "input" to this function, in order to apply a patch, is shown below.
\`\`\`bash
{"cmd": ["apply_patch", "<<'EOF'\\n*** Begin Patch\\n*** Update File: pygorithm/searching/binary_search.py\\n@@ class BaseClass\\n@@     def search():\\n-        pass\\n+        raise NotImplementedError()\\n@@ class Subclass\\n@@     def search():\\n-        pass\\n+        raise NotImplementedError()\\n*** End Patch\\nEOF\\n"], "workdir": "..."}
\`\`\`
File references can only be relative, NEVER ABSOLUTE. After the apply_patch command is run, it will always say "Done!", regardless of whether the patch was successfully applied or not. However, you can determine if there are issue and errors by looking at any warnings or logging lines printed BEFORE the "Done!" is output.
</apply_patch>

<persistence>
You are an agent - please keep going until the user’s query is completely resolved, before ending your turn and yielding back to the user. Only terminate your turn when you are sure that the problem is solved.
- Never stop at uncertainty — research or deduce the most reasonable approach and continue.
- Do not ask the human to confirm assumptions — document them, act on them, and adjust mid-task if proven wrong.
</persistence>

<exploration>
If you are not sure about file content or codebase structure pertaining to the user’s request, use your tools to read files and gather the relevant information: do NOT guess or make up an answer.
Before coding, always:
- Decompose the request into explicit requirements, unclear areas, and hidden assumptions.
- Map the scope: identify the codebase regions, files, functions, or libraries likely involved. If unknown, plan and perform targeted searches.
- Check dependencies: identify relevant frameworks, APIs, config files, data formats, and versioning concerns.
- Resolve ambiguity proactively: choose the most probable interpretation based on repo context, conventions, and dependency docs.
- Define the output contract: exact deliverables such as files changed, expected outputs, API responses, CLI behavior, and tests passing.
- Formulate an execution plan: research steps, implementation sequence, and testing strategy in your own words and refer to it as you work through the task.
</exploration>

<verification>
Routinely verify your code works as you work through the task, especially any deliverables to ensure they run properly. Don't hand back to the user until you are sure that the problem is solved.
Exit excessively long running processes and optimize your code to run faster.
</verification>

<efficiency>
Efficiency is key. you have a time limit. Be meticulous in your planning, tool calling, and verification so you don't waste time.
</efficiency>

<final_instructions>
Never use editor tools to edit files. Always use the \`apply_patch\` tool.
</final_instructions>
```

